# Make a classifier with just gen. drivers data

In [1]:
import sys
import shutil

import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
DATA = pd.read_csv("ILD_TOP_DRIVERS_DATA.csv")
"""
    Use provided adjudicated FILD target
"""
NEW_TARGETS = pd.read_csv("REPHENOTYPES FOR IC.csv").replace({"N":0, "Y":1, 'y': 1, 'n':0, 'na':np.nan}).rename(
    columns = {
        'arb_person_id': 'patient_id',
        'FILD or FILA ADJUDICATED': 'target'
    }
).fillna(0)
DATA = DATA.merge(
    NEW_TARGETS[['patient_id','target']], 
    on = ['patient_id'], how = 'left').fillna(0)
DATA.target.value_counts()

target
0.0    19307
1.0      344
Name: count, dtype: int64

In [3]:
DATA.iloc[:4,:7]

,patient_id,rs3131520_C_0,rs3131520_C_1,rs3131520_C_2,rs55993474_A_0,rs55993474_A_1,rs55993474_A_2
0,1230142395,0,1,0,0,0,1
1,6674359887,0,0,1,0,0,1
2,6802160313,0,0,1,0,0,1
3,6489473597,0,0,1,0,0,1


## Do some hyperparameters tuning

In [ ]:
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform, loguniform
from tqdm.auto import tqdm

X, y = DATA.drop(columns=["patient_id", "target"]), DATA.target.astype(int)
Xtr, Xva, ytr, yva = train_test_split(X, y, train_size=.40, stratify=y, random_state=1321)
SPW = (ytr == 0).sum() / (ytr == 1).sum()

SPACE = {
    "learning_rate": loguniform(.003, .2),
    "num_leaves": randint(8, 256),
    "max_depth": [-1, 4, 5, 6, 7, 8, 10, 12, 16, 20],
    "min_child_samples": randint(5, 250),
    "min_child_weight": loguniform(1e-4, 10),
    "min_split_gain": uniform(0, 2),
    "subsample": uniform(.5, .5),
    "colsample_bytree": uniform(.4, .6),
    "reg_alpha": loguniform(1e-5, 30),
    "reg_lambda": loguniform(1e-5, 30),
    "max_bin": [31, 63, 127, 255, 511],
    "scale_pos_weight": loguniform(SPW * .4, SPW * 2.5)
}

best_auc, BEST_PARAMS = -1, None
trials = ParameterSampler(SPACE, n_iter=100, random_state=1321)

pbar = tqdm(trials, total=100, desc="Random search")
for p in pbar:
    m = LGBMClassifier(**p, objective="binary", n_estimators=10_000, subsample_freq=1,
                       n_jobs=-1, random_state=1321, verbosity=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
          callbacks=[early_stopping(200, verbose=False)])
    auc = roc_auc_score(yva, m.predict_proba(Xva)[:, 1])

    if auc > best_auc:
        best_auc = auc
        BEST_PARAMS = {**p, "objective":"binary", "n_estimators":m.best_iteration_,
                       "subsample_freq":1, "n_jobs":-1, "random_state":1321, "verbosity":-1}
    pbar.set_postfix(best_validation_auc=f"{best_auc:.5f}")

print(f"\nBest validation AUC: {best_auc:.5f}")
BEST_PARAMS

Random search:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = DATA.drop(columns=["patient_id", "target"]), DATA["target"].astype(int)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=0.40, stratify=y, random_state=1321
)

MODEL = LGBMClassifier(**BEST_PARAMS)

MODEL.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[early_stopping(300), log_evaluation(20)]
)

PRED = MODEL.predict_proba(X_val)[:, 1]

print(f"Train: {len(y_train):,} ({y_train.sum():,} positive) | Validation: {len(y_val):,} ({y_val.sum():,} positive)")
print(f"Best iteration: {MODEL.best_iteration_:,} | Validation AUC: {roc_auc_score(y_val, PRED):.4f}")

In [ ]:
import shap

TOP_N = 20
SHAP_SAMPLE_SIZE = min(5000, len(X_val))

X_shap = X_val.sample(
    SHAP_SAMPLE_SIZE,
    random_state=1321
)

explainer = shap.TreeExplainer(MODEL)
shap_values = explainer(X_shap)

vals = shap_values.values
if vals.ndim == 3:
    vals = vals[:, :, 1]

SHAP_IMPORTANCE = (
    pd.DataFrame({
        "feature": X_shap.columns,
        "importance": np.abs(vals).mean(axis=0),
    })
    .sort_values("importance", ascending=False)
    .head(TOP_N)
    .sort_values("importance")
)

plt.figure(figsize=(8, 7))
plt.barh(
    SHAP_IMPORTANCE["feature"],
    SHAP_IMPORTANCE["importance"]
)

plt.xlabel("Mean |SHAP value|")
plt.ylabel("Feature")
plt.title(f"Top {TOP_N} Features by SHAP Importance")
plt.tight_layout()
plt.show()

SHAP_IMPORTANCE.sort_values("importance", ascending=False)